In [1]:
import copy
import io
import networkx as nx
import numpy as np
import pandas as pd
import sys
import warnings

from contextlib import redirect_stdout
from dowhy import CausalModel
from econml.inference import BootstrapInference
from networkx import DiGraph
from pathlib import Path
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.ensemble import GradientBoostingRegressor
from word2num import Word2Num

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_utils as graph_utils

from config_handlers.causal_handler import CausalHandler
from config_handlers.rk_handler import RKHandler
# from utils.vis import save_graph_to_img

warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniforge/base/envs/causal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "sharegpt4v"
DATASET = "vqav2"

In [3]:
rk_hdl = RKHandler()
rk_hdl.set_curr_model(MODEL)
rk_hdl.set_curr_ds(DATASET)
causal_hdl = CausalHandler()
causal_hdl.set_curr_ds(DATASET)
causal_hdl.set_curr_model(MODEL)

Load responses_counter
- These files already include the answer for the "complete RK"

In [4]:
# TODO: change this to Path(__file__).parent for script
base_dir = Path("..")
resps_path = base_dir.joinpath(rk_hdl.get_rk_final_path())
print(resps_path)
resps = data_io.load_jsonl(resps_path)

counter_resps_path = base_dir.joinpath(causal_hdl.get_counter_resps_path())
print(counter_resps_path)
alt_resps = data_io.load_jsonl(counter_resps_path)

../data/really_know/final/sharegpt4v/vqav2/results.jsonl
../data/causal_effects/counter_resps/sharegpt4v/vqav2/counter_resps.jsonl


In [5]:
# Get responses only for the max num of triplets
# (i.e., ignore the rest of the powerset for now)
resps_ids = set([q["question_id"] for q in resps])

# question_id: response
# this only gets the first response
dict_counter_resp = {
    q_idx: next(r["response"] for r in alt_resps if r["question_id"] == q_idx)
    for q_idx in resps_ids
}

For each question, add a new entry corresponding to the original response.

In [6]:
resps[0].keys()

dict_keys(['question_id', 'response', 'triple_objs'])

In [7]:
# Based on the counterfactual responses we have, pick and add the original ones.
all_resps = copy.deepcopy(alt_resps)
for q_idx in resps_ids:
    og_response = next((r for r in resps if r["question_id"] == q_idx), None)
    if og_response:
        all_resps.append(
            {
                "question_id": og_response["question_id"],
                "size": 0,
                "occluded": [],
                "response": og_response["response"],
            }
        )
    else:
        print(f"We have an issue with question {q_idx}.")

Create dataframe out of counterfactual data and responses

In [8]:
df = pd.DataFrame(all_resps)
df.drop(["path"], axis=1, inplace=True)

In [9]:
df.head()

,question_id,occluded,size,response
0,266922002,[36278-man],1,2
1,266922002,[new-sunglasses],1,2
2,266922002,[65128-watch],1,2
3,266922002,[5087-belt],1,2
4,266922002,[new-pants],1,2


Make ```response``` lowercase for consistency

In [10]:
df["response"] = df["response"].apply(lambda x: x.lower())

In [11]:
df[df["size"] == 0]

,question_id,occluded,size,response
1297,251824001,[],0,scissors
1298,224757004,[],0,8
1299,266922002,[],0,2
1300,332607000,[],0,1
1301,6719001,[],0,sitting
1302,391375002,[],0,sitting
1303,521306011,[],0,wii
1304,199215001,[],0,truck
1305,232692003,[],0,water skiing
1306,544857000,[],0,snowboarding


In [12]:
# Create mapping between questions and the biggest size of constraints.
dict_max_triplets = {
    q_idx: max([int(q["size"]) for q in all_resps if q["question_id"] == q_idx])
    for q_idx in resps_ids
}

# Make dictionary with {question_id: [concepts]}.
# This is done to extract an actual list of concepts, not any other PD objects.
dict_concepts = {
    q_idx: df[(df["question_id"] == q_idx) & (df["size"] == dict_max_triplets[q_idx])][
        "occluded"
    ].iloc[0]
    for q_idx in dict_max_triplets
}

Fix numerical answers given as words (e.g., "Three")

In [13]:
def fix_word_numbers(parser, x):
    try:
        res = int(x)
    except Exception as _:
        res = parser.parse(x)
        if not res:
            res = x  # If parse fails, it is probably a string
        else:
            res = int(res)  # If parse works, make sure an int is returned
    finally:
        return res


w2n = Word2Num()
df["response"] = df["response"].apply(lambda x: fix_word_numbers(w2n, x))

In [14]:
df.head()

,question_id,occluded,size,response
0,266922002,[36278-man],1,2
1,266922002,[new-sunglasses],1,2
2,266922002,[65128-watch],1,2
3,266922002,[5087-belt],1,2
4,266922002,[new-pants],1,2


Add column ```y``` representing the model answer
- If response is numeric, just use that
- If response is text, create lookup for the results of that question

In [15]:
lookup_y = {}

for idx, row in df.iterrows():
    q_idx = row["question_id"]
    if q_idx not in lookup_y:
        lookup_y[q_idx] = {}

    try:
        y = pd.to_numeric(row["response"])
        lookup_y[q_idx][row["response"]] = y
    except Exception as _:
        # Get all rows with same question ID
        curr_q = df[df["question_id"] == q_idx]
        # Create dict entry with list of unique responses
        lookup_y[q_idx] = {v: k for k, v in enumerate(curr_q["response"].unique())}
        y = lookup_y[q_idx][row["response"]]
    finally:
        df.loc[idx, "y"] = y

In [16]:
df.head()

,question_id,occluded,size,response,y
0,266922002,[36278-man],1,2,2.0
1,266922002,[new-sunglasses],1,2,2.0
2,266922002,[65128-watch],1,2,2.0
3,266922002,[5087-belt],1,2,2.0
4,266922002,[new-pants],1,2,2.0


In [17]:
# Make dictionary of dataframes w.r.t. question id
df_dict = {q_idx: df[df["question_id"] == q_idx] for q_idx in dict_counter_resp}

In [18]:
df_dict[266922002].head()

,question_id,occluded,size,response,y
0,266922002,[36278-man],1,2,2.0
1,266922002,[new-sunglasses],1,2,2.0
2,266922002,[65128-watch],1,2,2.0
3,266922002,[5087-belt],1,2,2.0
4,266922002,[new-pants],1,2,2.0


Making binary columns indicating whether a concept was occluded or not.
By default, all concepts are present (1). Then, we zero out the ones occluded.

Note: each data sample has a different amount of concepts 'attached' to it

In [19]:
# Adding 1 column (of zeros) to each sub-dataframe for each concept found
for q_idx in df_dict:
    for concept in dict_concepts[q_idx]:
        df_dict[q_idx][concept] = 1

In [20]:
df_dict[266922002].head()

,question_id,occluded,size,response,y,36278-man,new-sunglasses,65128-watch,5087-belt,new-pants,new-shoes,new-gloves,31995-jacket,15121-coat,29750-hat,46132-ponytail holder
0,266922002,[36278-man],1,2,2.0,1,1,1,1,1,1,1,1,1,1,1
1,266922002,[new-sunglasses],1,2,2.0,1,1,1,1,1,1,1,1,1,1,1
2,266922002,[65128-watch],1,2,2.0,1,1,1,1,1,1,1,1,1,1,1
3,266922002,[5087-belt],1,2,2.0,1,1,1,1,1,1,1,1,1,1,1
4,266922002,[new-pants],1,2,2.0,1,1,1,1,1,1,1,1,1,1,1


In [21]:
# Correcting the dataframes based on occlusions
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["occluded"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 0
    df_dict[q_idx].drop(["occluded"], axis=1, inplace=True)

View the different responses from the models

In [22]:
# for q_idx in df_dict:
#     unique_res = df_dict[q_idx]["response"].unique()
#     print(f"Question {q_idx} -> {len(unique_res)} unique responses {unique_res}")

In [23]:
df_dict[266922002].head()

,question_id,size,response,y,36278-man,new-sunglasses,65128-watch,5087-belt,new-pants,new-shoes,new-gloves,31995-jacket,15121-coat,29750-hat,46132-ponytail holder
0,266922002,1,2,2.0,0,1,1,1,1,1,1,1,1,1,1
1,266922002,1,2,2.0,1,0,1,1,1,1,1,1,1,1,1
2,266922002,1,2,2.0,1,1,0,1,1,1,1,1,1,1,1
3,266922002,1,2,2.0,1,1,1,0,1,1,1,1,1,1,1
4,266922002,1,2,2.0,1,1,1,1,0,1,1,1,1,1,1


```df_dict```
- Dictionary with question_ids as keys for which we have valid RKs
- Each entry in the dictionary is a dataframe with
  - The original response (no occlusion; size_treatment=0)
  - A bunch of rows with the counterfactual responses (combinations of concepts)
  - A series of binary columns matching 1:1 the concepts present or occluded

### Causal Analysis

We create the NX graphs based on the data contained from ```resps```. This is done so that we can create the ```.dot``` string required by the ```dowhy``` package.

In [ ]:
def rk_to_nx(raw_rk_rels: dict) -> DiGraph:
    rk_list = [
        {
            "from_concept": rk["from_concept"]["bb_label"]["bb_label_full"],
            "relationship": rk["rel_label"]["rel_label_text"],
            "to_concept": rk["to_concept"]["bb_label"]["bb_label_full"],
        }
        for rk in raw_rk_rels
    ]
    nx_graph = graph_utils.create_nx_graph(rk_list)

    # Removing self loops for causal analysis
    self_loops = list(nx.selfloop_edges(nx_graph))
    if self_loops:
        print(f"- Found self loops: {self_loops}.")
        nx_graph.remove_edges_from(nx.selfloop_edges(nx_graph))
    else:
        print("- No self-loops found.")
    return nx_graph


def make_nx_graphs(resps):
    graphs = {}
    for r in resps:
        graphs[r["question_id"]] = rk_to_nx(r["triple_objs"])
    return graphs


rk_graphs = make_nx_graphs(resps)
all_img_ids = list(rk_graphs.keys())
print(f"Loaded {len(rk_graphs)} from {MODEL} + {DATASET}")

Loaded 21 from sharegpt4v + vqav2


In [25]:
print(f"Duplicate ids: {set([x for x in all_img_ids if all_img_ids.count(x) > 1])}")

Duplicate ids: set()


In [26]:
# Add to each graph a node corresponding to the outcome variable 'y'
for rk in rk_graphs.values():
    rk.add_node("y", label="response")

    for node in list(rk.nodes()):
        if node == "y":
            continue
        rk.add_edge(node, "y")

Running causal analysis

Note to self: try generalised adjustment criterion (`general_adjustment` smth smth) instead of backdoor (`backdoor.econml.dml.DML`), if it exists...

Tutorial here: https://github.com/py-why/dowhy/blob/main/docs/source/example_notebooks/dowhy_generalized_covariate_adjustment_estimation_example.ipynb

In [27]:
def run_causal_inference_DML(
    data,
    nx_graph,
    test_significance=False,
    confidence_intervals=False,
    get_stderr=False,
    do_refute=False,
):
    all_estimates = {}
    all_refutations = {}

    # Prepare data for multinomial logistic regressor
    # responses = data["y"]  # 1-d vector of the responses
    cols_to_skip = ["question_id", "size", "response"]
    cols_to_keep = list(set(data.keys()) - set(cols_to_skip))
    data_for_estimation = data[cols_to_keep]  # n_observations x regressors
    regressors = list(data_for_estimation.drop("y", axis=1).keys())

    for col in regressors:
        print(f"Working on {col}...")

        # Exclude the current 'col' from the variables that can condition the estimates
        effect_modifiers = [r for r in regressors if r != col]

        # effect_modifiers: estimates are separately computed (conditioned)
        # for each value of the effect_modifier.
        model = CausalModel(
            data=data_for_estimation,
            treatment=col,
            outcome="y",
            graph=nx_graph,
            effect_modifiers=effect_modifiers,
        )
        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Define method params
        # If less than default n_splits (5), use only 1 split
        n_split = 2 if len(data) < 5 else 5
        method_params = {
            "init_params": {
                "model_y": GradientBoostingRegressor(),
                "model_t": GradientBoostingRegressor(),
                "model_final": LassoCV(fit_intercept=False, cv=n_split),
                "featurizer": PolynomialFeatures(degree=1, include_bias=False),
            },
            "fit_params": {},
        }
        if test_significance:
            method_params["fit_params"] = {
                "inference": BootstrapInference(n_bootstrap_samples=1, n_jobs=-1)
            }

        # Control and treatment values are correct since we
        # are occluding (removing) elements from images.
        ce_estimate = model.estimate_effect(
            estimand,
            method_name="backdoor.econml.dml.DML",
            control_value=1,
            treatment_value=0,
            test_significance=test_significance,
            confidence_intervals=confidence_intervals,
            fit_estimator=True,
            method_params=method_params,
        )
        # Compute standard error for the estimate
        if get_stderr:
            ce_estimate.get_standard_error()
        all_estimates[col] = ce_estimate

        # Run refutation with Placebo
        if do_refute:
            all_refutations[col] = model.refute_estimate(
                estimand,
                ce_estimate,
                method_name="placebo_treatment_refuter",
                show_progress_bar=False,
                placebo_type="permute",
            )

    return all_estimates, all_refutations

In [49]:
from utils.graph_stats import compute_ic
from utils.graph_utils import expand_cycles, load_vg1800_dict

sgg_dicts = load_vg1800_dict()

In [52]:
def handle_cycles(graph: DiGraph) -> DiGraph:
    # 1. Get first set of expanded cycles
    exp_cycles = expand_cycles(list(nx.simple_cycles(graph)))
    if not exp_cycles:
        print("No cycles found.")
        return graph

    print(f"Identified cycles: {exp_cycles}")
    while exp_cycles:
        # 2. Group these cycles based on their size
        idx_to_extract = 0
        exp_cycles_dict = {}
        for c in exp_cycles:
            if len(c) not in exp_cycles_dict:
                exp_cycles_dict[len(c)] = []
            exp_cycles_dict[len(c)].append(c)
        # 3. Consider smallest size and pop one
        min_size = min(list(exp_cycles_dict.keys()))
        curr_cycle = exp_cycles_dict[min_size][idx_to_extract]
        least_inf = {"u": "", "v": "", "ic": np.inf}
        for edge in curr_cycle:
            # 4. Get label for edges in that cycle
            label = graph.edges[edge[0], edge[1]]["label"]
            # 5. Compute information content of label
            ic = compute_ic(
                label, sgg_dicts["predicate_counts"], sgg_dicts["predicate_freqs"]
            )
            if ic < least_inf["ic"]:
                least_inf.update({"u": edge[0], "v": edge[1], "ic": ic})
        # 6. Remove edge corresponding to the min. information content
        graph.remove_edge(least_inf["u"], least_inf["v"])
        # 7. Re-compute expanded cycles within the current graph
        exp_cycles = expand_cycles(list(nx.simple_cycles(graph)))
    return graph

In [53]:
estimates = {}
refutations = {}

for q_idx, rk_data in rk_graphs.items():
    print(f">> Working with question # {q_idx}")
    curr_rk = rk_data.copy()
    curr_df = df_dict[q_idx]

    # Self-loops are handled during the creation of the NX objects
    # Handle cycles here
    curr_rk = handle_cycles(curr_rk)

    # Save graph image
    # curr_out_dir = f"{CE_OUT_DIR}/{MODEL}/{DATASET}"
    # make_dir(curr_out_dir)
    # save_graph_to_img(g_idx, graph, curr_out_dir)

    # Compute causal effects
    estimates[q_idx], refutations[q_idx] = run_causal_inference_DML(
        curr_df,
        curr_rk,
        test_significance=False,
        confidence_intervals=False,
        get_stderr=False,
        do_refute=False,
    )

>> Working with question # 266922002
No cycles found.
Working on 31995-jacket...
Working on new-gloves...
Working on 36278-man...
Working on 65128-watch...
Working on new-sunglasses...
Working on 5087-belt...
Working on 29750-hat...
Working on new-pants...
Working on 46132-ponytail holder...
Working on new-shoes...
Working on 15121-coat...
>> Working with question # 332607000
No cycles found.
Working on 16665-countertop...
Working on 68030-woman...
>> Working with question # 224757004
No cycles found.
Working on new-boys...
Working on new-jackets...
>> Working with question # 240275005
No cycles found.
Working on 58673-surfboard...
Working on 65479-wave...
>> Working with question # 249046003
No cycles found.
Working on 31995-jacket...
Working on 43809-person...
Working on new-gloves...
Working on 55627-snowboard...
Working on 30134-helmet...
Working on 62559-trick...
Working on 29750-hat...
Working on new-pants...
>> Working with question # 249883001
No cycles found.
Working on 43809-

Tests to resolve cycles in the RK graphs

Save causal estimates to file

In [28]:
causal_out_path = base_dir.joinpath(causal_hdl.get_estimates_path()).resolve()
print(causal_out_path)
data_io.make_dir(causal_out_path.parent)

/Users/lorenzocorti/Developer/mllm_diagnosis/data/causal_effects/estimates/sharegpt4v/vqav2/ce.jsonl
[Errno 17] File exists: '/Users/lorenzocorti/Developer/mllm_diagnosis/data/causal_effects/estimates/sharegpt4v/vqav2'


In [33]:
# Preparing data to save
def format_estimates(
    estimates, refutations=None, test_significance=False, std_error=False
):
    out_data = {}
    for q_idx in estimates:
        if q_idx not in out_data:
            out_data[q_idx] = {}

        for var_name, curr_est in estimates[q_idx].items():
            # var_name = curr_est._treatment_name[0]
            if var_name not in out_data[q_idx]:
                out_data[q_idx][var_name] = {}

            to_save = {
                "estimate": curr_est.value.item(),
                "estimand_type": curr_est.target_estimand.estimand_type.value,
                "estimand_expr": curr_est.realized_estimand_expr,
            }

            # Catch the output of the interpret() method
            f = io.StringIO()
            with redirect_stdout(f):
                curr_est.interpret()
            to_save["interpretation"] = f.getvalue().strip()

            if refutations:
                refute_data = refutations[q_idx][var_name]
                p_val = refute_data.refutation_result["p_value"]
                to_save["refute_test"] = {
                    "is_stat_significant": refute_data.refutation_result[
                        "is_statistically_significant"
                    ].item(),
                }
                if isinstance(p_val, tuple):
                    to_save["refute_test"]["p_value"] = list(p_val)
                elif isinstance(p_val, np.floating):
                    to_save["refute_test"]["p_value"] = p_val.item()

            if test_significance:
                p_val = curr_est.test_stat_significance()["p_value"]
                if isinstance(p_val, tuple):
                    to_save["p_value"] = list(p_val)
                elif isinstance(p_val, np.floating):
                    to_save["p_value"] = p_val.item()
                # to_save["significance"] = None

            if std_error:
                to_save["std_error"] = curr_est.get_standard_error()

            out_data[q_idx][var_name] = to_save
    return out_data


out_data = format_estimates(
    estimates, refutations=refutations, test_significance=True, std_error=True
)

In [34]:
out_data[266922002]

{'46132-ponytail holder': {'estimate': -0.08491006272019433,
  'estimand_type': 'nonparametric-ate',
  'estimand_expr': 'b: y~46132-ponytail holder+36278-man | 5087-belt,65128-watch,new-sunglasses,new-gloves,29750-hat,new-pants,new-shoes,15121-coat,31995-jacket',
  'interpretation': "Increasing the treatment variable(s) [46132-ponytail holder] from 1 to [0] causes an increase of -0.08491006272019433 in the expected value of the outcome [['y']], over the data distribution/population represented by the dataset.",
  'refute_test': {'is_stat_significant': False, 'p_value': 0.96},
  'p_value': [0, 0.001],
  'std_error': np.float64(0.015433661258678276)},
 '5087-belt': {'estimate': 0.01522084180489832,
  'estimand_type': 'nonparametric-ate',
  'estimand_expr': 'b: y~5087-belt+36278-man | 46132-ponytail holder,65128-watch,new-gloves,new-sunglasses,29750-hat,new-pants,new-shoes,15121-coat,31995-jacket',
  'interpretation': "Increasing the treatment variable(s) [5087-belt] from 1 to [0] causes 

In [39]:
# save
data_io.save_json(out_data, causal_out_path, indent=None)